# Dynamic Delta hedging and model-risk pressure

This study uses the merged M3/F1 interfaces to examine dynamic Black–Scholes replication on **model-generated pricing-measure paths**. It varies hedge cadence, volatility specification, and proportional transaction costs while preserving explicit seeds and accounting.

This is not a historical backtest, forecast, or trading-profit claim. See [`m3_dynamic_delta_hedging.md`](../docs/models/m3_dynamic_delta_hedging.md).


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from qf_platform.application import (
    HedgeWorkbenchConfig,
    HedgeWorkbenchRequest,
    canonical_black_scholes_draft,
    compose_black_scholes_study,
    run_hedge_workbench,
)

composition = compose_black_scholes_study(canonical_black_scholes_draft())


def run_condition(
    generating_volatility,
    hedging_volatility,
    transaction_cost_rate,
    *,
    replicate_count,
):
    return run_hedge_workbench(
        HedgeWorkbenchRequest(
            composition=composition,
            config=HedgeWorkbenchConfig(
                generating_volatility=generating_volatility,
                hedging_volatility=hedging_volatility,
                rebalance_day_interval=7,
                seed=0,
                replicate_count=replicate_count,
                transaction_cost_rate=transaction_cost_rate,
            ),
        )
    )


# The 64-seed correct-vs-misspecified comparison matches M3's committed fixture scale.
correct = run_condition(0.30, 0.30, 0.0, replicate_count=64)
misspecified = run_condition(0.30, 0.20, 0.0, replicate_count=64)
costly = run_condition(0.20, 0.20, 0.001, replicate_count=16)

In [ ]:
def show_table(headers, rows):
    text = "| " + " | ".join(headers) + " |\n"
    text += "| " + " | ".join("---" for _ in headers) + " |\n"
    for row in rows:
        text += "| " + " | ".join(str(value) for value in row) + " |\n"
    display(Markdown(text))


rows = []
for label, study in (
    ("correct volatility, frictionless", correct),
    ("hedging volatility 20% vs generating 30%", misspecified),
    ("20% volatility + 10 bp stock-trade cost", costly),
):
    summary = study.selected_summary
    mean_cost = sum(
        item.total_transaction_cost for item in study.selected_replicates
    ) / len(study.selected_replicates)
    rows.append(
        (
            label,
            summary.replicate_count,
            f"{summary.mean_error:.4f}",
            f"{summary.mean_absolute_error:.4f}",
            f"{summary.root_mean_square_error:.4f}",
            f"{mean_cost:.4f}",
        )
    )
show_table(
    (
        "condition",
        "replicates",
        "mean error",
        "MAE",
        "RMSE",
        "mean stock-trade cost",
    ),
    rows,
)

assert all(item.total_transaction_cost == 0.0 for item in correct.selected_replicates)
assert all(
    item.total_transaction_cost == 0.0 for item in misspecified.selected_replicates
)
assert all(item.total_transaction_cost > 0.0 for item in costly.selected_replicates)

In [ ]:
frequency = {
    point.rebalance_day_interval: point.summary for point in correct.frequency_evidence
}
assert frequency[1].root_mean_square_error < frequency[30].root_mean_square_error

intervals = sorted(frequency)
rmse = [frequency[interval].root_mean_square_error for interval in intervals]

plt.figure(figsize=(7, 4))
plt.plot(intervals, rmse, marker="o")
plt.xlabel("rebalance interval (calendar days)")
plt.ylabel("replication-error RMSE")
plt.title("Same path ensemble, different hedge cadence")
plt.show()

The path observation grid is held fixed while the rebalance schedule changes. A shorter hedge interval therefore addresses **discrete-rebalancing error**; it is not an SDE timestep-convergence experiment.


In [ ]:
labels = ["correct", "volatility misspecified", "transaction costs"]
rmse_values = [
    correct.selected_summary.root_mean_square_error,
    misspecified.selected_summary.root_mean_square_error,
    costly.selected_summary.root_mean_square_error,
]
assert (
    misspecified.selected_summary.root_mean_square_error
    > correct.selected_summary.root_mean_square_error
)

plt.figure(figsize=(7, 4))
plt.bar(labels, rmse_values)
plt.ylabel("replication-error RMSE")
plt.title("Different pressure sources should remain separately identified")
plt.xticks(rotation=15)
plt.show()

In [ ]:
correct_errors = [item.replication_error for item in correct.selected_replicates]
misspecified_errors = [
    item.replication_error for item in misspecified.selected_replicates
]

plt.figure(figsize=(8, 4))
plt.plot(correct_errors, label="correct volatility")
plt.plot(misspecified_errors, label="hedging volatility 20%, generating 30%")
plt.xlabel("matched seed index")
plt.ylabel("terminal replication error")
plt.title("Matched stochastic replicates")
plt.legend()
plt.show()

## Interpretation

The evidence separates three mechanisms: finite hedge cadence, volatility/model misspecification, and explicit transaction-cost drag. The paths are simulated under the model's pricing-measure semantics, so this study supports a **replication-mechanism stress test**, not historical trading profitability or physical-world forecasting. It also does not establish Heston hedge superiority; the repository intentionally has no authoritative Heston hedge comparison yet.
